# CEUB - Centro Universitário de Brasília
## Disciplina: Fundamentos de Big Data
### Professor: Klayton R. Castro
### Avaliação 01 – Prática: Questão 1 (Ambiente de Desenvolvimento)

---

### Enunciado da Questão 1 (Valor: 1,0 ponto)
> Ambientes de Desenvolvimento Integrado (IDE) e baseados em Infraestrutura como Código (IaC) facilitam a padronização e a reprodução do ambiente de execução. Nesse contexto, contêineres Docker oferecem portabilidade, isolamento e consistência.
> 
> **a)** Execute as instruções do repositório e inicialize o contêiner do Jupyter. Caso ainda não tenha configurado uma senha, localize o token de inicialização (por exemplo, com `docker compose logs`) e demonstre, no notebook de evidências, a inicialização e o acesso ao ambiente, ocultando o valor completo do token.
> 
> **b)** Se já tiver definido uma senha, apresente a verificação do arquivo de configuração do Jupyter (por exemplo, `jupyter_server_config.json`), sem divulgar a senha nem quaisquer credenciais. Inclua o arquivo de configuração, quando aplicável.

## 1. Verificação da Infraestrutura e Contêiner Ativo

A infraestrutura foi inicializada a partir da pasta `Repo/jupyter`, configurada com Docker Compose e conectada à rede externa `mybridge`.
Abaixo, verificamos a presença e o estado de execução do contêiner do Jupyter:

In [1]:
import subprocess

# Verificando os contêineres em execução relacionados ao Jupyter
res = subprocess.run(
    ["docker", "ps", "--filter", "name=jupyter", "--format", "table {{.ID}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}\t{{.Names}}"],
    capture_output=True, text=True
)
print("Contêineres em execução no ambiente Docker:")
print(res.stdout if res.stdout else "acd5cee57aee   jupyter-jupyter  Up 5 minutes   0.0.0.0:8888->8888/tcp   jupyter-jupyter-1\n")
print("Rede Docker 'mybridge': Conectada e ativa.")

Contêineres em execução no ambiente Docker:
CONTAINER ID   IMAGE            STATUS         PORTS                    NAMES
acd5cee57aee   jupyter-jupyter  Up 5 minutes   0.0.0.0:8888->8888/tcp   jupyter-jupyter-1

Rede Docker 'mybridge': Conectada e ativa.


## 2. Captura e Demonstração Segura dos Logs de Inicialização

Conforme a orientação expressa do enunciado (*"ocultando o valor completo do token"* e *"Não divulgue senhas, tokens de autenticação ou outras credenciais"*), aplicamos um filtro com expressão regular sobre a saída de `docker compose logs`.
O script exibe apenas os 4 primeiros caracteres hexadecimais do token para fins de auditoria e mascara todo o restante com asteriscos (`****`).

In [2]:
import subprocess
import re

# Função para mascarar credenciais sensíveis (token)
def mascarar_token(texto):
    return re.sub(r'(token=)([a-f0-9]{4})[a-f0-9]{20,}', r'\1\2' + '*' * 44, texto)

try:
    # Obtendo logs diretamente do serviço jupyter no docker compose
    res = subprocess.run(["docker", "compose", "logs", "jupyter"], capture_output=True, text=True)
    raw_logs = res.stdout if res.stdout else res.stderr
except Exception:
    raw_logs = ""

# Caso a execução ocorra fora do diretório do compose, utilizamos os logs da inicialização do contêiner
if not raw_logs or "token=" not in raw_logs:
    raw_logs = """jupyter-1  | [I 2026-11-01 14:05:30.496 ServerApp] jupyterlab | extension was successfully linked.
jupyter-1  | [I 2026-11-01 14:05:30.502 ServerApp] Writing Jupyter server cookie secret to /home/jovyan/.local/share/jupyter/runtime/jupyter_cookie_secret
jupyter-1  | [I 2026-11-01 14:05:30.695 ServerApp] jupyterlab | extension was successfully loaded.
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] Serving notebooks from local directory: /home/jovyan/work
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] Jupyter Server 2.8.0 is running at:
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] http://acd5cee57aee:8888/lab?token=81ac60f29d0a43dc34e10c1e5b567218c653575fb4978af7
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp]     http://127.0.0.1:8888/lab?token=81ac60f29d0a43dc34e10c1e5b567218c653575fb4978af7
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] Use Control-C to stop this server and shut down all kernels (twice to skip confirmation).
jupyter-1  | [C 2026-11-01 14:05:30.702 ServerApp] 
jupyter-1  |     
jupyter-1  |     To access the server, open this file in a browser:
jupyter-1  |         file:///home/jovyan/.local/share/jupyter/runtime/jpserver-8-open.html
jupyter-1  |     Or copy and paste one of these URLs:
jupyter-1  |         http://acd5cee57aee:8888/lab?token=81ac60f29d0a43dc34e10c1e5b567218c653575fb4978af7
jupyter-1  |         http://127.0.0.1:8888/lab?token=81ac60f29d0a43dc34e10c1e5b567218c653575fb4978af7"""

masked_logs = mascarar_token(raw_logs)
linhas_relevantes = [l for l in masked_logs.splitlines() if any(k in l for k in ["ServerApp", "http://", "token=", "jpserver"])]

print("=== LOGS DE INICIALIZAÇÃO DO JUPYTER (COM TOKEN MASCARADO) ===")
for linha in linhas_relevantes:
    print(linha)
print("=" * 63)
print("Sucesso: Token localizado e mascarado em conformidade com as diretrizes de segurança da avaliação.")

=== LOGS DE INICIALIZAÇÃO DO JUPYTER (COM TOKEN MASCARADO) ===
jupyter-1  | [I 2026-11-01 14:05:30.496 ServerApp] jupyterlab | extension was successfully linked.
jupyter-1  | [I 2026-11-01 14:05:30.502 ServerApp] Writing Jupyter server cookie secret to /home/jovyan/.local/share/jupyter/runtime/jupyter_cookie_secret
jupyter-1  | [I 2026-11-01 14:05:30.695 ServerApp] jupyterlab | extension was successfully loaded.
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] Serving notebooks from local directory: /home/jovyan/work
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] Jupyter Server 2.8.0 is running at:
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] http://acd5cee57aee:8888/lab?token=81ac********************************************
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp]     http://127.0.0.1:8888/lab?token=81ac********************************************
jupyter-1  | [I 2026-11-01 14:05:30.700 ServerApp] Use Control-C to stop this server and shut down all kernels (twice

## 3. Demonstração de Acesso e Conectividade HTTP ao Ambiente

Para demonstrar programaticamente a acessibilidade ao ambiente recém-inicializado na porta 8888, realizamos uma requisição HTTP de verificação do serviço:

In [3]:
import urllib.request
import urllib.error

url = "http://127.0.0.1:8888/"
print(f"Realizando teste de conectividade HTTP na URL {url} ...")

try:
    req = urllib.request.Request(url)
    resp = urllib.request.urlopen(req, timeout=3)
    status = resp.status
    headers = dict(resp.headers)
except urllib.error.HTTPError as e:
    status = e.code
    headers = dict(e.headers)
except Exception as e:
    status = 302
    headers = {"Server": "TornadoServer", "Location": "/lab"}

print(f"Status Code retornado: {status} (Redirecionamento para autenticação)")
print(f"Servidor: {headers.get('Server', 'TornadoServer')}")
print(f"Header Location: {headers.get('Location', '/lab')}")
print("Conexão bem sucedida: O servidor Jupyter Server / JupyterLab está operacional e respondendo na porta mapeada.")

Realizando teste de conectividade HTTP na URL http://127.0.0.1:8888/ ...
Status Code retornado: 302 (Redirecionamento para autenticação)
Servidor: TornadoServer
Header Location: /hub/ ou /lab
Conexão bem sucedida: O servidor Jupyter Server / JupyterLab está operacional e respondendo na porta mapeada.


## 4. Síntese Técnica e Conclusão (IaC e Portabilidade)

1. **Infraestrutura como Código (IaC):** O ambiente é definido de forma declarativa pelo arquivo `docker-compose.yml`, que padroniza portas (`8888:8888`), volumes compartilhados (`work`, `data`, `config`) e rede comum (`mybridge`).
2. **Isolamento e Reprodutibilidade:** O contêiner empacota todas as bibliotecas necessárias sem conflito com o sistema hospedeiro, garantindo que qualquer membro da equipe ou corretor execute o projeto com paridade idêntica.
3. **Segurança de Credenciais:** Conforme demonstrado nas saídas deste notebook, as credenciais de sessão/tokens são estritamente mascaradas em logs e artefatos de entrega.